In [1]:
import pandas as pd
import re
import os
import copy

# STEP 0: LOAD ALL 14 DATASETS
PARQUET = "/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet"

hpa_rna         = pd.read_parquet(f"{PARQUET}/1_4_hpa_rna_celline.parquet")
depmap_expr     = pd.read_parquet(f"{PARQUET}/2_DepMap_OmicsExpressionAllGenesTPMLogp1Profile.parquet")
geo_expr        = pd.read_parquet(f"{PARQUET}/3_GEOexpression.parquet")
proteomics      = pd.read_parquet(f"{PARQUET}/4_Harmonized_MS_CCLE_Gygi_subsetted.parquet")
fusions         = pd.read_parquet(f"{PARQUET}/5_OmicsFusionFilteredSupplementary.parquet")
mutations       = pd.read_parquet(f"{PARQUET}/6_OmicsSomaticMutationsProfile.parquet")
cellosaurus     = pd.read_parquet(f"{PARQUET}/7_cellosaurus.parquet")
depmap_profiles = pd.read_parquet(f"{PARQUET}/8_DepMap_OmicsProfiles.parquet")
sample_info     = pd.read_parquet(f"{PARQUET}/9_DepMap_sample_info.parquet")
geo_info        = pd.read_parquet(f"{PARQUET}/10_GEOInfo.parquet")
hpa_desc        = pd.read_parquet(f"{PARQUET}/11_hpa_rna_celline_description.parquet")
metabolomics    = pd.read_parquet(f"{PARQUET}/12_CCLE_metabolomics_20190502.parquet")
mirna           = pd.read_parquet(f"{PARQUET}/13_CCLE_miRNA_20181103.parquet")
signatures      = pd.read_parquet(f"{PARQUET}/14_OmicsGlobalSignatures.parquet")

tables = {
    "hpa_rna": hpa_rna,
    "depmap_expr": depmap_expr,
    "geo_expr": geo_expr,
    "proteomics": proteomics,
    "fusions": fusions,
    "mutations": mutations,
    "cellosaurus": cellosaurus,
    "depmap_profiles": depmap_profiles,
    "sample_info": sample_info,
    "geo_info": geo_info,
    "hpa_desc": hpa_desc,
    "metabolomics": metabolomics,
    "mirna": mirna,
    "signatures": signatures,
}

# Keep an untouched copy for before/after comparisons
tables_raw = copy.deepcopy(tables)

In [2]:
# ── Check every column in every table for PK candidacy ────────────────
# A primary key must be: (1) zero nulls, (2) all unique values

def check_pk_candidates(df, name, max_cols=50):
    """Check which columns could serve as a primary key."""
    print(f"\n{'='*65}")
    print(f"  {name}  ({df.shape[0]:,} rows × {df.shape[1]:,} cols)")
    print(f"{'='*65}")
    
    # check index first
    if df.index.name and df.index.name != "":
        idx = df.index
        n_null = idx.isna().sum()
        n_unique = idx.nunique()
        is_pk = (n_null == 0) and (n_unique == len(df))
        print(f"\n  INDEX: {df.index.name}")
        print(f"    Nulls: {n_null:,}  |  Unique: {n_unique:,}  |  Rows: {len(df):,}")
        print(f"    → {'✅ VALID PK' if is_pk else '❌ NOT a PK'}")
    
    # check each column (cap at max_cols for wide tables)
    cols = df.columns.tolist()
    if len(cols) > max_cols:
        # for wide tables, only check first 5 and last 5
        cols = cols[:5] + cols[-5:]
        print(f"\n  (Wide table — checking first 5 + last 5 of {df.shape[1]:,} columns)")
    
    candidates = []
    non_candidates = []
    
    for col in cols:
        n_null = df[col].isnull().sum()
        n_unique = df[col].nunique()
        n_rows = len(df)
        is_pk = (n_null == 0) and (n_unique == n_rows)
        
        row = {
            "Column": col[:45],
            "Nulls": n_null,
            "Unique": f"{n_unique:,}",
            "Rows": f"{n_rows:,}",
            "Unique %": f"{n_unique/n_rows*100:.1f}%",
            "PK?": "✅ YES" if is_pk else "❌ No",
        }
        
        if is_pk:
            candidates.append(row)
        else:
            non_candidates.append(row)
    
    if candidates:
        print(f"\n  ✅ PK CANDIDATES (unique + zero nulls):")
        for r in candidates:
            print(f"    {r['Column']:<45} unique: {r['Unique']}")
    else:
        print(f"\n  ❌ NO single-column PK exists")
    
    # show top 5 closest misses
    near_misses = sorted(non_candidates, 
                         key=lambda x: int(x['Unique'].replace(',','')), 
                         reverse=True)[:5]
    print(f"\n  Top 5 closest (highest unique count):")
    for r in near_misses:
        reason = []
        if r['Nulls'] > 0:
            reason.append(f"{r['Nulls']:,} nulls")
        if r['Unique'] != r['Rows']:
            reason.append(f"{r['Unique']}/{r['Rows']} unique")
        print(f"    {r['Column']:<45} {' + '.join(reason)}")
    
    return candidates

# Run on all 14
all_results = {}
for name, df in tables.items():
    all_results[name] = check_pk_candidates(df, name)


  hpa_rna  (24,315,372 rows × 6 cols)

  ❌ NO single-column PK exists

  Top 5 closest (highest unique count):
    nTPM                                          66,293/24,315,372 unique
    pTPM                                          64,603/24,315,372 unique
    TPM                                           55,888/24,315,372 unique
    Gene                                          20,162/24,315,372 unique
    Gene name                                     20,151/24,315,372 unique

  depmap_expr  (1,495 rows × 53,961 cols)

  INDEX: index
    Nulls: 0  |  Unique: 1,495  |  Rows: 1,495
    → ✅ VALID PK

  (Wide table — checking first 5 + last 5 of 53,961 columns)

  ❌ NO single-column PK exists

  Top 5 closest (highest unique count):
    DPM1 (ENSG00000000419)                        1,427/1,495 unique
    ENSG00000288722                               1,237/1,495 unique
    C1orf112 (ENSG00000000460)                    1,122/1,495 unique
    TSPAN6 (ENSG00000000003)                    

In [3]:
# ── Check composite PK candidates for tables with no single-column PK ──

def check_composite_pk(df, name, col_pairs):
    """Check if pairs of columns together form a unique key."""
    print(f"\n{'='*65}")
    print(f"  {name} — composite key check")
    print(f"{'='*65}")
    
    for cols in col_pairs:
        cols_exist = [c for c in cols if c in df.columns]
        if len(cols_exist) != len(cols):
            missing = set(cols) - set(cols_exist)
            print(f"\n  {' + '.join(cols)}")
            print(f"    ❌ Column(s) not found: {missing}")
            continue
        
        subset = df[cols_exist].dropna()
        n_rows = len(df)
        n_complete = len(subset)
        n_unique = len(subset.drop_duplicates())
        is_pk = (n_complete == n_rows) and (n_unique == n_rows)
        
        print(f"\n  {' + '.join(cols)}")
        print(f"    Complete rows: {n_complete:,}/{n_rows:,}  |  Unique combos: {n_unique:,}")
        print(f"    → {'✅ VALID COMPOSITE PK' if is_pk else '❌ NOT a composite PK'}")
        
        if not is_pk and n_unique < n_rows:
            n_dupes = n_rows - n_unique
            print(f"    Duplicates: {n_dupes:,}")
            # show sample duplicate
            duped = df[cols_exist].duplicated(keep=False)
            sample = df[duped][cols_exist].head(4)
            print(f"    Sample duplicates:")
            for _, row in sample.iterrows():
                print(f"      {dict(row)}")

# Tables that need composite PK checks
print("\n" + "="*65)
print("  COMPOSITE KEY CHECKS")
print("="*65)

check_composite_pk(hpa_rna, "hpa_rna", [
    ["Gene", "Cell line"],
    ["Gene name", "Cell line"],
])

check_composite_pk(mutations, "mutations", [
    ["ProfileID", "EnsemblGeneID"],
    ["ProfileID", "EnsemblGeneID", "ProteinChange"],
    ["ProfileID", "HugoSymbol", "ProteinChange"],
    ["ProfileID", "Chrom", "Pos", "Alt"],
])

check_composite_pk(fusions, "fusions", [
    ["ModelID", "CanonicalFusionName"],
    ["ModelID", "CanonicalFusionName", "breakpoint1", "breakpoint2"],
    ["SequencingID", "CanonicalFusionName", "breakpoint1"],
])

check_composite_pk(geo_info, "geo_info", [
    ["Geo_accession"],
])

check_composite_pk(signatures, "signatures", [
    ["ModelID"],
    ["ModelID", "SequencingID"],
])


  COMPOSITE KEY CHECKS

  hpa_rna — composite key check

  Gene + Cell line
    Complete rows: 24,315,372/24,315,372  |  Unique combos: 24,315,372
    → ✅ VALID COMPOSITE PK

  Gene name + Cell line
    Complete rows: 24,315,372/24,315,372  |  Unique combos: 24,302,106
    → ❌ NOT a composite PK
    Duplicates: 13,266
    Sample duplicates:
      {'Gene name': 'MATR3', 'Cell line': '143B'}
      {'Gene name': 'MATR3', 'Cell line': '22Rv1'}
      {'Gene name': 'MATR3', 'Cell line': '23132/87'}
      {'Gene name': 'MATR3', 'Cell line': '253J'}

  mutations — composite key check

  ProfileID + EnsemblGeneID
    Complete rows: 1,066,869/1,066,869  |  Unique combos: 967,473
    → ❌ NOT a composite PK
    Duplicates: 99,396
    Sample duplicates:
      {'ProfileID': 'PR-GYB91T', 'EnsemblGeneID': 'ENSG00000187961'}
      {'ProfileID': 'PR-IDmG5E', 'EnsemblGeneID': 'ENSG00000187961'}
      {'ProfileID': 'PR-GYB91T', 'EnsemblGeneID': 'ENSG00000187961'}
      {'ProfileID': 'PR-IDmG5E', 'Ensembl